- Cost = (input tokens × input price) + (output tokens × output price)
- gpt-5.5 and 5.4
    - latency = fast
    - max output = 128k tokens
    - context window = 1M
- gpt-5.4-mini
    - latency = faster
    - max output = 128k tokens
    - context window = 400k 

In [2]:
# cost
gpt_54_mini_input = 0.75
gpt_54_mini_output = 4.50

gpt_54_input = 2.5
gpt_54_output = 15

gpt_55_input = 5
gpt_55_output = 30

In [5]:
from openai import OpenAI
import os

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

openai_client = OpenAI(api_key=OPENAI_API_KEY)

# test cost calculation
def llm(user_prompt, system_prompt=None, model="gpt-5.4-mini"):
    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response

In [6]:
system_prompt = "Je bent een schrijver."
user_prompt = "Schrijf een verhaaltje over vliegende gehaktballetjes."

llm_response = llm(user_prompt=user_prompt, system_prompt=system_prompt)

In [14]:
print(llm_response.model_dump().keys())

dict_keys(['id', 'created_at', 'error', 'incomplete_details', 'instructions', 'metadata', 'model', 'object', 'output', 'parallel_tool_calls', 'temperature', 'tool_choice', 'tools', 'top_p', 'background', 'completed_at', 'conversation', 'max_output_tokens', 'max_tool_calls', 'previous_response_id', 'prompt', 'prompt_cache_key', 'prompt_cache_retention', 'reasoning', 'safety_identifier', 'service_tier', 'status', 'text', 'top_logprobs', 'truncation', 'usage', 'user', 'billing', 'frequency_penalty', 'moderation', 'presence_penalty', 'store'])


In [18]:
from pprint import pprint

pprint(llm_response.model_dump())

{'background': False,
 'billing': {'payer': 'developer'},
 'completed_at': 1780407634.0,
 'conversation': None,
 'created_at': 1780407629.0,
 'error': None,
 'frequency_penalty': 0.0,
 'id': 'resp_0fd101392c4417ce006a1edd4d42c081a39cf57ea33c062214',
 'incomplete_details': None,
 'instructions': None,
 'max_output_tokens': None,
 'max_tool_calls': None,
 'metadata': {},
 'model': 'gpt-5.4-mini-2026-03-17',
 'moderation': None,
 'object': 'response',
 'output': [{'content': [{'annotations': [],
                          'logprobs': [],
                          'text': 'Op een winderige dinsdag in het dorpje '
                                  'Kroketdam gebeurde iets heel bijzonders. In '
                                  'de keuken van mevrouw De Koning stond een '
                                  'grote pan te pruttelen. Ze had net '
                                  'gehaktballetjes gemaakt voor de soep, maar '
                                  'toen ze een beetje te veel peper toev

In [40]:
input_cost = (llm_response.usage.input_tokens / 1000000) * gpt_54_mini_input
output_cost = (llm_response.usage.output_tokens / 1000000) * gpt_54_mini_output
total_cost = input_cost + output_cost 
total_cost

0.0021202499999999997

In [37]:
# get summary text
summary = llm_response.output_text
# try with a summary task
summary_system_prompt = "Your role is to summarize all provided text."
summary_user_prompt = summary
llm_summary = llm(user_prompt=summary_user_prompt, system_prompt=summary_system_prompt)

In [38]:
pprint(llm_summary.model_dump())

{'background': False,
 'billing': {'payer': 'developer'},
 'completed_at': 1780409483.0,
 'conversation': None,
 'created_at': 1780409481.0,
 'error': None,
 'frequency_penalty': 0.0,
 'id': 'resp_0fff0f67cbc54c47006a1ee489837081908c9ef7c02a1794c4',
 'incomplete_details': None,
 'instructions': None,
 'max_output_tokens': None,
 'max_tool_calls': None,
 'metadata': {},
 'model': 'gpt-5.4-mini-2026-03-17',
 'moderation': None,
 'object': 'response',
 'output': [{'content': [{'annotations': [],
                          'logprobs': [],
                          'text': 'Op een winderige dinsdag in Kroketdam '
                                  'gingen de gehaktballetjes van mevrouw De '
                                  'Koning door te veel peper ineens zweven en '
                                  'vlogen ze door het dorp. De bewoners waren '
                                  'verbaasd, de kat Snor keek toe, en '
                                  'buurjongen Pim verzon een theorie over '

In [39]:
input_cost = (llm_summary.usage.input_tokens / 1000000) * gpt_54_mini_input
output_cost = (llm_summary.usage.output_tokens / 1000000) * gpt_54_mini_output
total_cost = input_cost + output_cost 
total_cost

0.00090975

In [41]:
type(total_cost)

float

# Generalize it

In [ ]:
MODEL_PRICING_PER_1M = {
    "gpt-5.4-mini": {
        "input": 0.75,
        "output": 4.50,
    },
    "gpt-5.4": {
        "input": 2.5,
        "output": 15,
    },
    "gpt-5.5": {
        "input": 5,
        "output": 30,
    }
}

In [ ]:
def calculate_token_cost(usage, model:str) -> tuple[float,float,float]:
    if model not in MODEL_PRICING_PER_1M:
        raise ValueError(f"No pricing configured for model: {model}")

    pricing = MODEL_PRICING_PER_1M[model]

    input_tokens = usage.input_tokens
    output_tokens = usage.output_tokens

    input_cost = (usage.input_tokens / 1000000) * pricing["input"]
    output_cost = (usage.output_tokens / 1000000) * pricing["output"]
    total_cost = input_cost + output_cost 
    return total_cost, input_cost, output_cost

